In [52]:
import pandas as pd
results = pd.read_csv('results.csv')
shootouts = pd.read_csv('shootouts.csv')
goalscorers = pd.read_csv('goalscorers.csv')
former_names = pd.read_csv('former_names.csv')


In [53]:
results.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0,0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4,2,Friendly,London,England,False
2,1874-03-07,Scotland,England,2,1,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2,2,Friendly,London,England,False
4,1876-03-04,Scotland,England,3,0,Friendly,Glasgow,Scotland,False


In [56]:
# Clean results.csv 
# Drop duplicates
results = results.drop_duplicates()

# Handle missing values
results = results.dropna(subset=['home_team', 'away_team', 'home_score', 'away_score'])

# Convert date column to datetime
results['date'] = pd.to_datetime(results['date'], errors='coerce')
results['year'] = results['date'].dt.year
results['month'] = results['date'].dt.month
results['era'] = (results['year'] // 10) * 10

# Clean team names (strip spaces and unify casing)
results['home_team'] = results['home_team'].str.strip().str.title()
results['away_team'] = results['away_team'].str.strip().str.title()

# Optional: remove invalid score values
results = results[(results['home_score'] >= 0) & (results['away_score'] >= 0)]
results['match_key'] = (
    results['date'].dt.strftime('%Y-%m-%d') + '|' +
    results['home_team'] + '|' + results['away_team'])

print("Cleaned Results Dataset:")
print(results.head())


Cleaned Results Dataset:
        date home_team away_team  home_score  away_score tournament     city  \
0 1872-11-30  Scotland   England           0           0   Friendly  Glasgow   
1 1873-03-08   England  Scotland           4           2   Friendly   London   
2 1874-03-07  Scotland   England           2           1   Friendly  Glasgow   
3 1875-03-06   England  Scotland           2           2   Friendly   London   
4 1876-03-04  Scotland   England           3           0   Friendly  Glasgow   

    country  neutral  year  month   era                    match_key  
0  Scotland    False  1872     11  1870  1872-11-30|Scotland|England  
1   England    False  1873      3  1870  1873-03-08|England|Scotland  
2  Scotland    False  1874      3  1870  1874-03-07|Scotland|England  
3   England    False  1875      3  1870  1875-03-06|England|Scotland  
4  Scotland    False  1876      3  1870  1876-03-04|Scotland|England  


In [57]:
# Clean former_names.csv
# Drop duplicates
former_names = former_names.drop_duplicates()

# Clean team name columns
former_names['current'] = former_names['current'].str.strip().str.title()
former_names['former'] = former_names['former'].str.strip().str.title()

# Convert date columns
former_names['start_date'] = pd.to_datetime(former_names['start_date'], errors='coerce')
former_names['end_date'] = pd.to_datetime(former_names['end_date'], errors='coerce')

# Handle missing values
former_names = former_names.dropna(subset=['current', 'former'])

print("Cleaned Former Names Dataset:")
print(former_names.head())


Cleaned Former Names Dataset:
          current                former start_date   end_date
0           Benin               Dahomey 1959-11-08 1975-11-30
1    Burkina Faso           Upper Volta 1960-04-14 1984-08-04
2         Curaçao  Netherlands Antilles 1957-03-03 2010-10-10
3  Czechoslovakia               Bohemia 1903-04-05 1919-01-01
4  Czechoslovakia   Bohemia And Moravia 1939-01-01 1945-05-01


In [58]:
# Create mapping dictionary
name_mapping = dict(zip(former_names['former'], former_names['current']))

# 3️⃣ Replace team names in results
for col in ['home_team', 'away_team']:
    results[col] = results[col].replace(name_mapping)

# 4️⃣ (Optional) Replace country names if you want current country versions
results['country'] = results['country'].replace(name_mapping)

# 5️⃣ Quick check
print("✅ Sample after name replacement:")
print(results[['date', 'home_team', 'away_team', 'country']].head(10))
print(results[['date', 'home_team', 'away_team', 'country']].tail(10))

✅ Sample after name replacement:
        date home_team away_team   country
0 1872-11-30  Scotland   England  Scotland
1 1873-03-08   England  Scotland   England
2 1874-03-07  Scotland   England  Scotland
3 1875-03-06   England  Scotland   England
4 1876-03-04  Scotland   England  Scotland
5 1876-03-25  Scotland     Wales  Scotland
6 1877-03-03   England  Scotland   England
7 1877-03-05     Wales  Scotland     Wales
8 1878-03-02  Scotland   England  Scotland
9 1878-03-23  Scotland     Wales  Scotland
            date      home_team     away_team        country
48356 2025-06-24         Canada   El Salvador  United States
48357 2025-06-24         Panama       Jamaica  United States
48358 2025-06-24     Guadeloupe     Guatemala  United States
48359 2025-06-28         Panama      Honduras  United States
48360 2025-06-28         Mexico  Saudi Arabia  United States
48361 2025-06-29         Canada     Guatemala  United States
48362 2025-06-29  United States    Costa Rica  United States
48363 

In [59]:
# Clean shootouts.csv
# Drop duplicates
shootouts = shootouts.drop_duplicates()

# Convert date to datetime
shootouts['date'] = pd.to_datetime(shootouts['date'], errors='coerce')

# Clean text columns
shootouts['winner'] = shootouts['winner'].str.strip().str.title()

# Handle missing data
shootouts = shootouts.dropna(subset=['date', 'winner'])
shootouts['match_key'] = (
    shootouts['date'].dt.strftime('%Y-%m-%d') + '|' +
    shootouts['home_team'].str.strip().str.title() + '|' +
    shootouts['away_team'].str.strip().str.title())

print("Cleaned Shootouts Dataset:")
print(shootouts.head())


Cleaned Shootouts Dataset:
        date    home_team         away_team       winner first_shooter  \
0 1967-08-22        India            Taiwan       Taiwan           NaN   
1 1971-11-14  South Korea  Vietnam Republic  South Korea           NaN   
2 1972-05-07  South Korea              Iraq         Iraq           NaN   
3 1972-05-17     Thailand       South Korea  South Korea           NaN   
4 1972-05-19     Thailand          Cambodia     Thailand           NaN   

                                 match_key  
0                  1967-08-22|India|Taiwan  
1  1971-11-14|South Korea|Vietnam Republic  
2              1972-05-07|South Korea|Iraq  
3          1972-05-17|Thailand|South Korea  
4             1972-05-19|Thailand|Cambodia  


In [60]:
print(shootouts.columns)


Index(['date', 'home_team', 'away_team', 'winner', 'first_shooter',
       'match_key'],
      dtype='object')


In [61]:
# Clean goalscorers.csv
# Drop duplicates
goalscorers = goalscorers.drop_duplicates()

# Convert date column
goalscorers['date'] = pd.to_datetime(goalscorers['date'], errors='coerce')

# Clean player and team names
goalscorers['scorer'] = goalscorers['scorer'].str.strip().str.title()
goalscorers['team'] = goalscorers['team'].str.strip().str.title()

# Handle missing values
goalscorers = goalscorers.dropna(subset=['scorer', 'team', 'date'])
goalscorers['match_key'] = (
    goalscorers['date'].dt.strftime('%Y-%m-%d') + '|' +
    goalscorers['team'])

print("Cleaned Goalscorers Dataset:")
print(goalscorers.head())


Cleaned Goalscorers Dataset:
        date  home_team away_team       team            scorer  minute  \
0 1916-07-02      Chile   Uruguay    Uruguay   José Piendibene    44.0   
1 1916-07-02      Chile   Uruguay    Uruguay  Isabelino Gradín    55.0   
2 1916-07-02      Chile   Uruguay    Uruguay  Isabelino Gradín    70.0   
3 1916-07-02      Chile   Uruguay    Uruguay   José Piendibene    75.0   
4 1916-07-06  Argentina     Chile  Argentina     Alberto Ohaco     2.0   

   own_goal  penalty             match_key  
0     False    False    1916-07-02|Uruguay  
1     False    False    1916-07-02|Uruguay  
2     False    False    1916-07-02|Uruguay  
3     False    False    1916-07-02|Uruguay  
4     False    False  1916-07-06|Argentina  


In [62]:
# Convert the date column to datetime
results['date'] = pd.to_datetime(results['date'], errors='coerce', dayfirst=True)

# Drop any rows where the date couldn't be parsed (optional)
results = results.dropna(subset=['date'])

# Format all dates as yyyy/mm/dd
results['date'] = results['date'].dt.strftime('%Y/%m/%d')

# Quick check
print("✅ Date format standardized:")
print(results[['date']].head(10))


✅ Date format standardized:
         date
0  1872/11/30
1  1873/03/08
2  1874/03/07
3  1875/03/06
4  1876/03/04
5  1876/03/25
6  1877/03/03
7  1877/03/05
8  1878/03/02
9  1878/03/23


In [63]:
results.to_csv('cleaned_results.csv', index=False)
shootouts.to_csv('cleaned_shootouts.csv', index=False)
goalscorers.to_csv('cleaned_goalscorers.csv', index=False)
former_names.to_csv('cleaned_former_names.csv', index=False)
